# 02 — Feature Analysis

**OandaFX** — Run the feature engineering pipeline, inspect distributions, correlations, and NaN analysis.

**Saves processed feature Parquets to Google Drive** for direct loading by the training notebook.

In [ ]:
# ─── Mount Google Drive & Setup ─────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/oandafx'
    if not os.path.exists('/content/oandafx'):
        !git clone https://github.com/your-org/oandafx.git /content/oandafx
    sys.path.insert(0, '/content/oandafx')
    !pip install -q pandas-ta
else:
    DRIVE_BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, DRIVE_BASE)

RAW_DIR = os.path.join(DRIVE_BASE, 'data', 'raw')
PROCESSED_DIR = os.path.join(DRIVE_BASE, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f'Raw data: {RAW_DIR}')
print(f'Processed output: {PROCESSED_DIR}')

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.storage import load_candles
from src.features.engineer import FeatureEngineer

plt.style.use('dark_background')
sns.set_palette('Set2')

In [ ]:
# ─── Load Raw Data ───────────────────────────────────────────────
data_path = Path(RAW_DIR)
pairs = sorted([d.name for d in data_path.iterdir() if d.is_dir()]) if data_path.exists() else []
print(f'Pairs: {pairs}')

# Initialize feature engineer
engineer = FeatureEngineer(
    base_timeframe='M15',
    higher_timeframes=['H1', 'H4', 'D'],
    lookback_bars=128,
    include_volume=True,
    include_patterns=True,
    include_mtf=True,
)

In [ ]:
# ─── Process All Pairs ───────────────────────────────────────────
feature_datasets = {}

for pair in pairs:
    print(f'\nProcessing {pair}...')
    raw_data = {}
    for tf in ['M15', 'H1', 'H4', 'D']:
        path = data_path / pair / f'{tf}.parquet'
        if path.exists():
            raw_data[tf] = load_candles(path)
            print(f'  {tf}: {len(raw_data[tf]):,} bars')
        else:
            print(f'  {tf}: MISSING')
    
    if 'M15' not in raw_data:
        print(f'  Skipping {pair} — no M15 data')
        continue
    
    features_df = engineer.build(raw_data)
    features_df['pair'] = pair
    feature_datasets[pair] = features_df
    print(f'  → {features_df.shape[0]:,} rows × {features_df.shape[1]} features')

print(f'\nTotal pairs processed: {len(feature_datasets)}')

## Feature Distribution Analysis

In [ ]:
# ─── Combine and Inspect ─────────────────────────────────────────
all_features = pd.concat(feature_datasets.values(), ignore_index=True)
print(f'Combined dataset: {all_features.shape[0]:,} rows × {all_features.shape[1]} columns')

# Get feature columns
feature_cols = engineer.get_feature_columns(all_features)
print(f'Feature dimensions: {len(feature_cols)}')
print(f'\nSample feature names: {feature_cols[:20]}')

In [ ]:
# ─── NaN Analysis ────────────────────────────────────────────────
nan_pct = all_features[feature_cols].isna().mean().sort_values(ascending=False)
nan_nonzero = nan_pct[nan_pct > 0]

if not nan_nonzero.empty:
    print(f'Features with NaN values: {len(nan_nonzero)}')
    display(nan_nonzero.head(20))
else:
    print('✅ No NaN values in feature columns (after warmup drop)')

In [ ]:
# ─── Feature Distributions (histograms for key features) ────────
key_features = [
    'return_1', 'rsi_14', 'macd_hist', 'adx_14',
    'bb_pct_b', 'vol_regime', 'mtf_alignment', 'stoch_k',
]
key_present = [f for f in key_features if f in all_features.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, feat in zip(axes.flat, key_present):
    data = all_features[feat].dropna()
    ax.hist(data, bins=80, alpha=0.7, color='cyan')
    ax.set_title(feat, fontsize=11)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=0.8)

for ax in axes.flat[len(key_present):]:
    ax.set_visible(False)

plt.suptitle('Key Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Feature Correlation Heatmap (top 30 features) ──────────────
top_features = feature_cols[:30]
corr = all_features[top_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, fmt='.1f',
            annot=False, linewidths=0.5, square=True)
ax.set_title('Feature Correlation Matrix (top 30)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Feature Statistics Summary ──────────────────────────────────
stats = all_features[feature_cols].describe().T
stats['nan_pct'] = all_features[feature_cols].isna().mean()
stats['range'] = stats['max'] - stats['min']
print(f'Feature statistics ({len(feature_cols)} features):')
display(stats.head(30))

## Save Processed Features to Google Drive

In [ ]:
# ─── Save per-pair processed features ────────────────────────────
for pair, df in feature_datasets.items():
    out_path = os.path.join(PROCESSED_DIR, f'{pair}_M15_features.parquet')
    df.to_parquet(out_path, index=False)
    print(f'Saved {pair}: {len(df):,} rows → {out_path}')

# Save combined dataset
combined_path = os.path.join(PROCESSED_DIR, 'all_pairs_features.parquet')
all_features.to_parquet(combined_path, index=False)
print(f'\n✅ Combined dataset saved: {len(all_features):,} rows → {combined_path}')

# Save feature column list
import json
cols_path = os.path.join(PROCESSED_DIR, 'feature_columns.json')
with open(cols_path, 'w') as f:
    json.dump(feature_cols, f)
print(f'Feature columns ({len(feature_cols)}) saved → {cols_path}')